# 03 · Hypothesis Testing for Model Comparison

## What this notebook covers
Bootstrap CIs tell you the uncertainty around a metric. Hypothesis tests tell you whether an observed difference is likely to be real. This notebook covers the three tests most used in ML evaluation.

## The test menu
| Test | Data requirement | Null hypothesis |
|------|-----------------|----------------|
| **McNemar's** | Binary predictions on same test set | Models make same mistakes |
| **Wilcoxon signed-rank** | Paired continuous scores (e.g. per-fold AUC) | Symmetric distribution of differences |
| **Paired t-test** | Paired scores, approximately normal | Mean difference = 0 |

## Why McNemar's is preferred for classifiers
McNemar's uses only the *discordant* pairs — samples where models disagree. It is the most powerful test when you have a fixed test set and binary predictions. The Wilcoxon test is useful when you have repeated k-fold scores and don't want to assume normality.

## The multiple comparisons trap
Every additional model you add to a comparison multiplies your false positive rate. With 8 models and uncorrected α=0.05 you expect ~1.4 false discoveries purely by chance. Notebook 04 covers corrections.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar
from scipy import stats
import itertools
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(n_samples=1500, n_features=20, n_informative=12, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

models = {
    "GBM": GradientBoostingClassifier(n_estimators=100, random_state=0),
    "RF" : RandomForestClassifier(n_estimators=100, random_state=0),
    "LR" : LogisticRegression(max_iter=1000, random_state=0),
}
preds, probas = {}, {}
for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    preds[name]  = clf.predict(X_te)
    probas[name] = clf.predict_proba(X_te)[:, 1]
    print(f"{name} AUC: {roc_auc_score(y_te, probas[name]):.4f}")

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar's test between two classifiers. Returns (statistic, p_value)."""
    b = ((pred_a == y_true) & (pred_b != y_true)).sum()  # A right, B wrong
    c = ((pred_a != y_true) & (pred_b == y_true)).sum()  # A wrong, B right
    table = np.array([[0, b], [c, 0]])  # McNemar uses b, c cells
    result = mcnemar(table, exact=True)
    return result.statistic, result.pvalue

# Pairwise McNemar across all 3 models
model_names = list(preds.keys())
rows = []
for a, b in itertools.combinations(model_names, 2):
    stat, p = mcnemar_test(y_te, preds[a], preds[b])
    rows.append({"Pair": f"{a} vs {b}", "McNemar stat": f"{stat:.2f}", "p-value": f"{p:.4f}", "Significant (α=0.05)": "Yes" if p < 0.05 else "No"})

print(pd.DataFrame(rows).to_string(index=False))

## Wilcoxon signed-rank test on cross-validation AUC scores
When you have k-fold scores (one per fold, paired across models), Wilcoxon is the non-parametric alternative to a paired t-test.

In [ ]:
# Generate 10-fold CV AUC scores for each model
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
cv_aucs = {}
for name, clf in models.items():
    scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")
    cv_aucs[name] = scores
    print(f"{name} 10-fold AUC: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Wilcoxon signed-rank: GBM vs RF on fold AUC scores
stat_w, p_w = stats.wilcoxon(cv_aucs["GBM"], cv_aucs["RF"])
print(f"Wilcoxon GBM vs RF: stat={stat_w:.2f}, p={p_w:.4f}")
print(f"Conclusion: {'Significant difference' if p_w < 0.05 else 'No significant difference'} (α=0.05)")

# Paired t-test for comparison
stat_t, p_t = stats.ttest_rel(cv_aucs["GBM"], cv_aucs["RF"])
print(f"Paired t-test GBM vs RF: stat={stat_t:.2f}, p={p_t:.4f}")

## Summary
- Use **McNemar's** when comparing binary predictions on a fixed test set
- Use **Wilcoxon** when comparing continuous fold-level scores without normality assumption
- Use **paired t-test** when you have many folds (≥30) and can invoke CLT
- All three tests assume *paired* data — never compare models trained or evaluated on different datasets

Next: when you run all pairwise comparisons simultaneously, you need multiple comparisons correction (Notebook 04).